In [37]:
%matplotlib inline
import torch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from matplotlib import pyplot as plt
import numpy as np
import random
from torch import nn
import torch.utils.data as Data

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']  # 使用黑体
plt.rcParams['axes.unicode_minus'] = False    # 解决负号显示问题

In [38]:
num_inputs = 2
num_examples = 1000
true_w = [2, -3.4]
true_b = 4.2
features = torch.tensor(np.random.normal(0, 1, (num_examples, num_inputs)), dtype=torch.float)
labels = true_w[0] * features[:, 0] + true_w[1] * features[:, 1] + true_b
labels += torch.tensor(np.random.normal(0, 0.01, size=labels.size()), dtype=torch.float)

print(features[0])
print(labels[0])

tensor([ 1.9547, -0.4816])
tensor(9.7514)


In [39]:
batch_size = 10
dataset = Data.TensorDataset(features, labels)
data_iter = Data.DataLoader(dataset, batch_size, shuffle=True)

In [40]:
for X,y in data_iter:
    print(X)
    print(y)
    break

tensor([[-0.4051,  0.1266],
        [ 0.7573, -0.7713],
        [-0.9231,  0.0830],
        [-1.0977, -0.6728],
        [ 0.4220,  0.6569],
        [-0.2730,  0.5233],
        [-0.3725,  1.3185],
        [ 1.2161,  2.2041],
        [ 0.4477,  0.1661],
        [-1.0178, -0.6968]])
tensor([ 2.9554,  8.3318,  2.0707,  4.2935,  2.8020,  1.8870, -1.0242, -0.8709,
         4.5303,  4.5505])


In [41]:
class LinearNet(nn.Module):
    def __init__(self, n_feature):
        super(LinearNet, self).__init__()
        self.linear = nn.Linear(n_feature, 1)
    def forward(self, x):
        y = self.linear(x)
        return y

net = LinearNet(num_inputs)
print(net)

LinearNet(
  (linear): Linear(in_features=2, out_features=1, bias=True)
)


In [42]:
for param in net.parameters():
    print(param)

Parameter containing:
tensor([[-0.5401, -0.0489]], requires_grad=True)
Parameter containing:
tensor([-0.3217], requires_grad=True)


In [43]:
nn.init.normal_(net.linear.weight, mean=0, std=0.01)
nn.init.constant_(net.linear.bias, val=0)

Parameter containing:
tensor([0.], requires_grad=True)

In [44]:
loss = nn.MSELoss()

In [45]:
import torch.optim as optim

optimizer = optim.SGD(net.parameters(), lr=0.03)
print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.03
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [46]:
num_epochs = 3
for epoch in range(1, num_epochs + 1):
    for X, y in data_iter:
        output = net(X)
        l = loss(output, y.view(-1, 1))
        optimizer.zero_grad()
        l.backward()
        optimizer.step()
    print('epoch %d, loss: %f' % (epoch, l.item()))

epoch 1, loss: 0.000121
epoch 2, loss: 0.000077
epoch 3, loss: 0.000068


In [48]:
dense = net.linear
print(true_w, dense.weight)
print(true_b, dense.bias)

[2, -3.4] Parameter containing:
tensor([[ 1.9996, -3.3999]], requires_grad=True)
4.2 Parameter containing:
tensor([4.1992], requires_grad=True)
